## Importations
- codecs pour les encodages
- pandas et numpy pour les calculs sur tableaux
- matplotlib pour les graphiques
- itertools pour les itérateurs sophistiqués (paires sur liste, ...)

In [27]:
# -*- coding: utf8 -*-
import codecs,glob
import features
import re,os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import itertools as it
import pickle
#%pylab inline
#pd.options.display.mpl_style = 'default'
debug=False
from __future__ import print_function
from IPython.display import display
def ding():
    os.system('afplay /System/Library/Sounds/Submarine.aiff')

In [28]:
pd.__version__
%store -r num
# num=0
print(num)

0


In [29]:
phonologicalMap="-X"
sample="%d"%num

In [30]:
rep="/Volumes/BroadExt/sDrive/Recherche/Boye/HDR/Data/vlexique2.0.3/WordStructure/"
fTirages="vlexique2-S%s.csv"%sample

In [31]:
import math
def rAn(r,n):
    f = math.factorial
    return f(n) / f(n-r)
def rCn(r,n):
    f = math.factorial
    return f(n) / f(r) / f(n-r)

### Préparation des matrices de traits

In [32]:
features.add_config('/Users/gilles/Github/SWIM/ParadigmGeneration/Vlexique2/bdlexique.ini')
fs=features.FeatureSystem('phonemes')

In [33]:
validPhonemes=list(fs.supremum.concept.extent)
for phoneme in validPhonemes:
    print (phoneme, [phoneme], ";", end=" ")

p ['p'] ; t ['t'] ; k ['k'] ; b ['b'] ; d ['d'] ; g ['g'] ; f ['f'] ; s ['s'] ; S ['S'] ; v ['v'] ; z ['z'] ; Z ['Z'] ; m ['m'] ; n ['n'] ; J ['J'] ; N ['N'] ; j ['j'] ; l ['l'] ; r ['r'] ; w ['w'] ; H ['H'] ; i ['i'] ; y ['y'] ; E ['E'] ; e ['e'] ; 9 ['9'] ; 2 ['2'] ; 6 ['6'] ; a ['a'] ; u ['u'] ; O ['O'] ; o ['o'] ; ê ['ê'] ; û ['û'] ; â ['â'] ; ô ['ô'] ; 

In [34]:
neutralisationsNORD=(u"6û",u"9ê")
neutralisationsSUD=(u"e2o",u"E9O")
if phonologicalMap=="-N":
    neutralisations=neutralisationsNORD
elif phonologicalMap=="-S":
    neutralisations=neutralisationsSUD
else:
    neutralisations=(u"",u"")
    phonologicalMap=("-X")
bdlexiqueIn = u"èò"+neutralisations[0]
bdlexiqueNum = [ord(char) for char in bdlexiqueIn]
neutreOut = u"EO"+neutralisations[1]
neutralise = dict(zip(bdlexiqueNum, neutreOut))

In [35]:
def recoder(chaine,table=neutralise):
    if type(chaine)==str:
        temp=chaine.translate(table)
        result=temp
    elif type(chaine)==unicode:
        result=chaine.translate(table)
    else:
        result=chaine
    return result

### Vérification de la phonotactique des glides du français
- si *prononciation* est *None* renvoyer *None*
- ajout de diérèses dans les séquences mal-formées
- vérification des séquences consonne+glide à la finale

In [36]:
dierese={"j":"ij", "w":"uw","H":"yH","i":"ij","u":"uw","y":"yH"}

In [37]:
def checkFrench(prononciation):
    if prononciation and not pd.isnull(prononciation):
        result=recoder(prononciation)
        m=re.match(r"^.*([^ieèEaOouy926êôâ])[jwH]$",result)
        if m:
            print ("pb avec un glide final", [prononciation])
        m=re.match(r"(.*[ptkbdgfsSvzZ][rl])([jwH])(.*)",result)
        if m:
            n=re.search(r"[ptkbdgfsSvzZ][rl](wa|Hi|wê)",result)
            if not n:
                glide=m.group(2)
                result=m.group(1)+dierese[glide]+m.group(3)
        m=re.match(r"(.*)([iuy])([ieEaOouy].*)",result)
        if m:
            glide=m.group(2)
            result=m.group(1)+dierese[glide]+m.group(3)
    else:
        result=prononciation
    return result

In [38]:
checkFrench("ye")

'yHe'

# Lecture du tableau de VERBES

### Lecture du lexique
- nomLexique pour le fichier
- names pour les noms de colonnes
- élimination des lignes dupliquées éventuelles (p.e. dépendre)

In [39]:
nomLexique=rep+fTirages
paradigmes=pd.read_csv(nomLexique,sep=";")
paradigmes.head()

,lexeme,ai1P,ai1S,ai2P,ai2S,ai3P,ai3S,fi1P,fi1S,fi2P,...,ppFP,ppFS,ppMP,ppMS,ps1P,ps1S,ps2P,ps2S,ps3P,ps3S
0,abaisser,NaN,abEsE,abEsat,NaN,abEsEr,abEsa,abEs6rô,abEs6rE,abEs6re,...,abEse,abEse,abEse,abEse,abEsjô,abEs,NaN,abEs,abEs,abEs
1,abandonner,abâdOnam,abâdOnE,abâdOnat,abâdOna,abâdOnEr,abâdOna,abâdOn6rô,abâdOn6rE,abâdOn6re,...,abâdOne,abâdOne,abâdOne,abâdOne,abâdOnjô,abâdOn,abâdOnje,abâdOn,abâdOn,abâdOn
2,abasourdir,NaN,NaN,NaN,abazurdi,NaN,NaN,NaN,abazurdirE,NaN,...,abazurdi,abazurdi,abazurdi,abazurdi,NaN,NaN,NaN,NaN,NaN,abazurdis
3,abattre,NaN,NaN,NaN,abati,abatir,abati,abatrô,abatrE,abatre,...,abaty,abaty,abaty,abaty,abatjô,abat,abatje,abat,abat,abat
4,abdiquer,NaN,NaN,NaN,NaN,NaN,abdika,abdik6rô,abdik6rE,abdik6re,...,NaN,NaN,abdike,abdike,NaN,abdik,NaN,abdik,NaN,abdik


In [40]:
casesPrincipales= [
        'inf', 'pi1S', 'pi2S', 'pi3S', 'pi1P', 'pi2P', 'pi3P', 'ii1S',
        'ii2S', 'ii3S', 'ii1P', 'ii2P', 'ii3P', 
        'fi1S', 'fi2S', 'fi3S', 'fi1P', 'fi2P',
        'fi3P', 'pI2S', 'pI1P', 'pI2P', 'ps1S', 'ps2S', 'ps3S', 'ps1P',
        'ps2P', 'ps3P', 
        'pc1S', 'pc2S', 'pc3S', 'pc1P', 'pc2P', 'pc3P', 'pP',
        'ppMS', 'ppMP', 'ppFS', 'ppFP'
            ]
casesSecondaires= [
       'ai1S', 'ai2S', 'ai3S', 'ai1P', 'ai2P', 'ai3P', 'is1S', 'is2S', 'is3S', 'is1P', 'is2P', 'is3P'
            ]
casesTotales=casesPrincipales+casesSecondaires
listeCases=casesTotales

### Suppression de la colonne index inutile

In [41]:
if u"Unnamed: 0" in paradigmes:
    del paradigmes[u"Unnamed: 0"]
paradigmes.columns

Index(['lexeme', 'ai1P', 'ai1S', 'ai2P', 'ai2S', 'ai3P', 'ai3S', 'fi1P',
       'fi1S', 'fi2P', 'fi2S', 'fi3P', 'fi3S', 'ii1P', 'ii1S', 'ii2P', 'ii2S',
       'ii3P', 'ii3S', 'inf', 'is1P', 'is1S', 'is2P', 'is2S', 'is3P', 'is3S',
       'pI1P', 'pI2P', 'pI2S', 'pP', 'pc1P', 'pc1S', 'pc2P', 'pc2S', 'pc3P',
       'pc3S', 'pi1P', 'pi1S', 'pi2P', 'pi2S', 'pi3P', 'pi3S', 'ppFP', 'ppFS',
       'ppMP', 'ppMS', 'ps1P', 'ps1S', 'ps2P', 'ps2S', 'ps3P', 'ps3S'],
      dtype='object')

In [42]:
sampleCases=paradigmes.columns.tolist()
sampleCases.remove(u"lexeme")
print(",".join(sampleCases))
print("nombre de cases du paradigme de l'échantillon :",len(sampleCases))

ai1P,ai1S,ai2P,ai2S,ai3P,ai3S,fi1P,fi1S,fi2P,fi2S,fi3P,fi3S,ii1P,ii1S,ii2P,ii2S,ii3P,ii3S,inf,is1P,is1S,is2P,is2S,is3P,is3S,pI1P,pI2P,pI2S,pP,pc1P,pc1S,pc2P,pc2S,pc3P,pc3S,pi1P,pi1S,pi2P,pi2S,pi3P,pi3S,ppFP,ppFS,ppMP,ppMS,ps1P,ps1S,ps2P,ps2S,ps3P,ps3S
nombre de cases du paradigme de l'échantillon : 51


### Application de la neutralisation phonologique et stockage des paradigmes neutralisés correspondants

In [43]:
#Adapt all the forms to French phonology
for case in sampleCases:
    paradigmes[case]=paradigmes[case].apply(lambda x: checkFrench(x))

# Préparation du calcul des analogies

### Calcul de la différence entre deux formes

In [44]:
def diff(mot1,mot2):
    result=[]
    diff1=""
    diff2=""
    same=""
    vide="."
    lmax=max(len(mot1),len(mot2))
    lmin=min(len(mot1),len(mot2))
    for index in range(lmax):
        if index < lmin:
            if mot1[index]!=mot2[index]:
                diff1+=mot1[index]
                diff2+=mot2[index]
                same+=vide
            else:
                same+=mot1[index]
                diff1+=vide
                diff2+=vide
        elif index < len(mot1):
            diff1+=mot1[index]
        elif index < len(mot2):
            diff2+=mot2[index]
    diff1=diff1.lstrip(".")
    diff2=diff2.lstrip(".")
#    return (same,diff1,diff2,diff1+"_"+diff2)
    return (diff1+"-"+diff2)

### Accumulation des paires appartenant à un patron

In [45]:
def rowDiff(row, patrons):
    result=diff(row[0],row[1])
    if not result in patrons:
        patrons[result]=(formesPatron(),formesPatron())
    patrons[result][0].ajouterFormes(row[0])
    patrons[result][1].ajouterFormes(row[1])
    return (result[0],result[1])

### Transformation d'un patron en RegExp

In [46]:
def patron2regexp(morceaux):
    result="^"
    for morceau in morceaux:
        if morceau=="*":
            result+="(.*)"
        elif len(morceau)>1:
            result+="(["+morceau+"])"
        else:
            result+=morceau
    result+="$"
    result=result.replace(")(","")
    return result

### Substitution de sortie 
???

In [47]:
def remplacementSortie(sortie):
    n=1
    nsortie=""
    for lettre in sortie:
        if lettre==".":
            nsortie+="\g<%d>"%n
            n+=1
        else:
            nsortie+=lettre
    return nsortie

In [48]:
class formesPatron:
    '''
    Accumulateur de formes correspondant à un patron pour calcul de la Généralisation Minimale (cf. MGL)
    '''
    def __init__(self):
        self.formes=[]

#    def __repr__(self):
#        return ','.join(self.calculerGM())
        
    def ajouterForme(self,forme):
        self.formes.append(forme)
        
    def calculerGM(self):
        minLongueur=len(min(self.formes, key=len))
        maxLongueur=len(max(self.formes, key=len))
        if debug: 
            print (minLongueur, maxLongueur, file=logfile)
            print (minLongueur, maxLongueur)
        positions=[]
        if maxLongueur>minLongueur:
            positions.append("*")
        for i in xrange(minLongueur, 0, -1):
            phonemes=set([x[-i] for x in self.formes])
            # print("phonemes",phonemes)
            if debug: 
                print (phonemes, file=logfile)
                print (phonemes)
            if "." in phonemes:
                positions.append(".")
            else:
                positions.append("".join(fs.lattice[phonemes].extent))
        return patron2regexp(positions)

class pairePatrons:
    '''
    Accumulateur de triplets (f1,f2,patron) correspondant à une paire pour calcul des Généralisations Minimales (cf. MGL)
    '''
    def __init__(self,case1,case2):
        self.patrons1={}
        self.patrons2={}
        self.case1=case1
        self.case2=case2

#    def __repr__(self):
#        return ','.join(self.calculerGM())
        
    def ajouterFormes(self,forme1,forme2,patron):
#        print (forme1,forme2,patron, file=logfile)
        patron12=patron
        (pat1,pat2)=patron.split("-")
        patron21=pat2+"-"+pat1
#        print (patron12,patron21, file=logfile)
        if not patron12 in self.patrons1:
            self.patrons1[patron12]=formesPatron()
        self.patrons1[patron12].ajouterForme(forme1)
        if not patron21 in self.patrons2:
            self.patrons2[patron21]=formesPatron()
        self.patrons2[patron21].ajouterForme(forme2)
        
        
    def calculerGM(self):
        resultat1={}
        for patron in self.patrons1:
            if debug: 
                print ("patron1", patron, file=logfile)
                print ("patron1", patron)
            resultat1[patron]=self.patrons1[patron].calculerGM()
        resultat2={}
        for patron in self.patrons2:
            if debug: 
                print ("patron2", patron, file=logfile)
                print ("patron2", patron)
            resultat2[patron]=self.patrons2[patron].calculerGM()
        return (resultat1,resultat2) 

# Classe pour la gestion des patrons, des classes et des transformations

In [49]:
class paireClasses:
    def __init__(self,case1,case2):
        self.case1=case1
        self.case2=case2
        self.nom=case1+"-"+case2
        self.classes1=classesPaire(case1,case2)
        self.classes2=classesPaire(case2,case1)

    def ajouterPatron(self,n,patron,motif):
        if n==1:
            self.classes1.ajouterPatron(patron,motif)
        elif n==2:
            self.classes2.ajouterPatron(patron,motif)
        else:
            if debug: print ("le numéro de forme n'est pas dans [1,2]",n, file=logfile)

    def ajouterPaire(self,forme1,forme2):
        self.classes1.ajouterPaire(forme1,forme2)
        self.classes2.ajouterPaire(forme2,forme1)
        
    def calculerClasses(self):
        return(self.classes1,self.classes2)

    
class classesPaire:
    '''
    Gestion des patrons, des classes et des transformations
    
    ajouterPatron : ajoute un patron et son motif associé (MGL)
    ajouterPaire : ajoute une paire de formes, calcule la classe de la forme1 et la règle sélectionnée
    sortirForme : cacule les formes de sortie correspondant à la forme1 avec leurs coefficients respectifs
    '''
    def __init__(self,case1,case2):
        self.case1=case1
        self.case2=case2
        self.nom=case1+"-"+case2
        self.classe={}
        self.nbClasse={}
        self.patrons={}
        self.entree={}
        self.sortie={}
        self.classeCF={}
        self.nbClasseCF={}
    
    def ajouterPatron(self,patron,motif):
        self.patrons[patron]=motif
        (entree,sortie)=patron.split("-")
        self.entree[patron]=entree.replace(u".",u"(.)")
        self.sortie[patron]=remplacementSortie(sortie)
    
    def ajouterPaire(self,forme1,forme2):
        '''
        on calcule la classe de la paire idClasseForme et la règle sélectionnée
        on incrémente le compteur de la classe et celui de la règle sélectionnée à l'intérieur de la classe
        '''
        classeFormeCF=[]
        regleFormeCF=""
        classeForme=[]
        regleForme=""
        for patron in self.patrons:
            filterF1=".*"+patron.split("-")[0]+"$"
            if re.match(filterF1,forme1):
                classeFormeCF.append(patron)
                if forme2==re.sub(self.entree[patron]+"$",self.sortie[patron],forme1):
                    regleFormeCF=patron
            filterF1=self.patrons[patron]
            if re.match(filterF1,forme1):
                classeForme.append(patron)
                '''
                le +"$" permet de forcer l'alignement à droite pour les transformations suffixales
                '''
                if forme2==re.sub(self.entree[patron]+"$",self.sortie[patron],forme1):
                    regleForme=patron
        idClasseFormeCF=", ".join(classeFormeCF)
        if not idClasseFormeCF in self.classeCF:
            self.classeCF[idClasseFormeCF]={}
            self.nbClasseCF[idClasseFormeCF]=0
        if not regleFormeCF in self.classeCF[idClasseFormeCF]:
            self.classeCF[idClasseFormeCF][regleFormeCF]=0
        self.nbClasseCF[idClasseFormeCF]+=1
        self.classeCF[idClasseFormeCF][regleFormeCF]+=1
        
        idClasseForme=", ".join(classeForme)
        if not idClasseForme in self.classe:
            self.classe[idClasseForme]={}
            self.nbClasse[idClasseForme]=0
        if not regleForme in self.classe[idClasseForme]:
            self.classe[idClasseForme][regleForme]=0
        self.nbClasse[idClasseForme]+=1
        self.classe[idClasseForme][regleForme]+=1

    def sortirForme(self,forme,contextFree=False):
        classeForme=[]
        sortieForme={}
        for patron in self.patrons:
            if contextFree:
                filterF1=".*"+patron.split("-")[0]+"$"
            else:
                filterF1=self.patrons[patron]
            if re.match(filterF1,forme):
                classeForme.append(patron)
        if classeForme:
            idClasseForme=", ".join(classeForme)
            if contextFree:
                nbClasse=self.nbClasseCF
                classe=self.classeCF
            else:
                nbClasse=self.nbClasse
                classe=self.classe
            if idClasseForme in nbClasse:
                nTotal=nbClasse[idClasseForme]
                for patron in classe[idClasseForme]:
                    sortie=re.sub(self.entree[patron]+"$",self.sortie[patron],forme)
                    sortieForme[sortie]=float(classe[idClasseForme][patron])/nTotal
            else:
                if debug: 
                    print (forme, file=logfile)
                    print ("pas de classe",idClasseForme, file=logfile)
                    print ("%.2f par forme de sortie" % (float(1)/len(classeForme)), file=logfile)
                nTotal=len(classeForme)
                for patron in classeForme:
                    sortie=re.sub(self.entree[patron]+"$",self.sortie[patron],forme)
                    sortieForme[sortie]=float(1)/nTotal
        else:
            if debug:
                print (forme, file=logfile) 
                print ("pas de patron", file=logfile)
        return sortieForme
        

## Appliquer la formule de calcul des différences entre chaines à chaque ligne

>si il y a au moins une ligne

>>on applique la différence à la ligne

>>on calcule les deux patrons par suppression des points initiaux

>>on renvoie le groupement par patrons (1&2)

>sinon

>>on renvoie le paradigme vide d'origine

In [50]:
def rapports(paradigme):
    (lexeme,case1,case2)= paradigme.columns.values.tolist()
    patrons=pairePatrons(case1,case2)
    classes=paireClasses(case1,case2)
    # if case1 in [u'ai3S', u'inf'] and case2 in [u'ai3S', u'inf']:
    #     print (patrons.patrons1)
    #     print (patrons.patrons2)
        # print (classes)
    if len(paradigme)>0:
        paradigme.apply(lambda x: patrons.ajouterFormes(x.iloc[1],x.iloc[2],diff(x.iloc[1],x.iloc[2])), axis=1)
        (regles1,regles2)=patrons.calculerGM()
        for regle in regles1:
            classes.ajouterPatron(1,regle,regles1[regle])
        for regle in regles2:
            classes.ajouterPatron(2,regle,regles2[regle])
        paradigme.apply(lambda x: classes.ajouterPaire(x.iloc[1],x.iloc[2]), axis=1)
    (classes1,classes2)=classes.calculerClasses()
    return (classes1,classes2)

### Dédoubler les lignes avec des surabondances dans *colonne*
>identifier une ligne avec surabondance

>>ajouter les lignes correspondant à chaque valeur

>>ajouter le numéro de la ligne initiale dans les lignes à supprimer

>supprimer les lignes avec surabondance

NB : il faut préparer le tableau pour avoir une indexation qui permette l'ajout des valeurs individuelles et la suppression des lignes de surabondances

In [51]:
def splitCellMates(df,colonne):
    '''
    Calcul d'une dataframe sans surabondance par dédoublement des valeurs
    '''
    dfPaire=df.copy()
    indexCols=dfPaire.columns.tolist()
    indexCols.remove(colonne)
    # print(indexCols)
    dfPaire=dfPaire.set_index(indexCols).apply(lambda x: x.str.split(',').explode()).reset_index()
    return dfPaire[indexCols+[colonne]]


## Calculer les rapports entre formes pour chaque paire

>on fait la liste des cases de *paradigmes*

>pour chaque paire du tableau principal

>>si la paire fait partie des cases de *paradigmes*

>>>on calcule le rapport

>>sinon

>>>on signale que qu'une des cases n'est pas représentée

In [52]:
def evaluerEchantillon(paradigmes):
    result={}
    colonnes=paradigmes.columns.values.tolist()
    for n,paire in enumerate(it.combinations_with_replacement(sampleCases,2)):
        # progressBar.value=n
        print (paire)
        if debug: print (paire, file=logfile)
        if debug: print ("-".join(paire),end=", ")
        paireListe=list(paire)
        paireListe.insert(0,"lexeme")
        if paire[0] in colonnes and paire[1] in colonnes:
            paradigmePaire=paradigmes[paireListe].dropna(thresh=3, axis=0).reindex()
            if paire[0]==paire[1]:
                paradigmePaire=paradigmes[paireListe].dropna(thresh=2, axis=0).reindex()
                paradigmePaire.columns=["lexeme", paire[0],paire[0]+"-bis"]
                paradigmePaire=splitCellMates(splitCellMates(paradigmePaire,paire[0]),paire[0]+"-bis")
                paradigmePaire.columns=["lexeme", paire[0],paire[0]]
                # if paire[0]=="inf": print(paradigmePaire.to_string())
            else:
                paradigmePaire=splitCellMates(splitCellMates(paradigmePaire,paire[0]),paire[1])
            # display(paradigmePaire)
            result[paire]=rapports(paradigmePaire)
            # if paire[0]=="inf" and paire[1]=="inf":
                # (a,b)=result[paire]
                # print(a.patrons)
                # print(b.patrons)
        else:
            result[paire]=("missing pair", paire)
    return result

### Boucle de calcul des analogies pour l'échantillon

In [53]:
%%time

# in python3 xrange a changé de nom pour range.
xrange=range

debug=False
debug1=False
resultats=evaluerEchantillon(paradigmes)

('ai1P', 'ai1P')
('ai1P', 'ai1S')
('ai1P', 'ai2P')
('ai1P', 'ai2S')
('ai1P', 'ai3P')
('ai1P', 'ai3S')
('ai1P', 'fi1P')
('ai1P', 'fi1S')
('ai1P', 'fi2P')
('ai1P', 'fi2S')
('ai1P', 'fi3P')
('ai1P', 'fi3S')
('ai1P', 'ii1P')
('ai1P', 'ii1S')
('ai1P', 'ii2P')
('ai1P', 'ii2S')
('ai1P', 'ii3P')
('ai1P', 'ii3S')
('ai1P', 'inf')
('ai1P', 'is1P')
('ai1P', 'is1S')
('ai1P', 'is2P')
('ai1P', 'is2S')
('ai1P', 'is3P')
('ai1P', 'is3S')
('ai1P', 'pI1P')
('ai1P', 'pI2P')
('ai1P', 'pI2S')
('ai1P', 'pP')
('ai1P', 'pc1P')
('ai1P', 'pc1S')
('ai1P', 'pc2P')
('ai1P', 'pc2S')
('ai1P', 'pc3P')
('ai1P', 'pc3S')
('ai1P', 'pi1P')
('ai1P', 'pi1S')
('ai1P', 'pi2P')
('ai1P', 'pi2S')
('ai1P', 'pi3P')
('ai1P', 'pi3S')
('ai1P', 'ppFP')
('ai1P', 'ppFS')
('ai1P', 'ppMP')
('ai1P', 'ppMS')
('ai1P', 'ps1P')
('ai1P', 'ps1S')
('ai1P', 'ps2P')
('ai1P', 'ps2S')
('ai1P', 'ps3P')
('ai1P', 'ps3S')
('ai1S', 'ai1S')
('ai1S', 'ai2P')
('ai1S', 'ai2S')
('ai1S', 'ai3P')
('ai1S', 'ai3S')
('ai1S', 'fi1P')
('ai1S', 'fi1S')
('ai1S', 'fi2P')


In [54]:
classesFinales={}
for resultat in resultats:
    classesFinales[resultat]=resultats[resultat][0]
    classesFinales[(resultat[1],resultat[0])]=resultats[resultat][1]

In [55]:
with open(rep+fTirages.replace(".csv","-Regles.pkl"), 'wb') as output:
   pickle.dump(classesFinales, output, pickle.HIGHEST_PROTOCOL)

In [56]:
if num<8:
    num+=1
    ding()
else:
    num=0
    ding()
    ding()
    ding()
# %store num

# Fin du traitement

In [57]:
paireTest=("inf","inf")
formeTest=u"yHe"

In [58]:
classesFinales[paireTest].sortirForme(formeTest,contextFree=False)

{'yHe': 1.0}

In [59]:
classesFinales[paireTest].patrons

{'-': '^(.*[ptkbdgfsSvzZmnJNjlrwHiyEe926auOoêûâô][ptkbdgfsSvzZmnJNjlrwHiyEe926auOoêûâô][rEe6])$'}